# Kaggriculture: Abstraction Level 1 Training Pipeline
### 3-Phase Curriculum: Behavioral Cloning $\to$ Ghost-Play RL $\to$ Self-Play PPO League

This notebook implements the complete training progression for **Abstraction Level 1** (1-turn parameterized operations) utilizing the **Delta Net Worth ($\Delta\text{NW}$)** reward engine.

#### Architectural Workflow
1. **Phase 1: Supervised Behavioral Cloning (Offline)** — Pretrains Actor-Critic on winning replay trajectories via masked cross-entropy and final coin MSE.
2. **Phase 2: Ghost-Play Reinforcement Learning** — Uses PPO against historical Grandmaster replay opponents on matching episode seeds with $\Delta\text{NW}$ step rewards.
3. **Phase 3: True Self-Play PPO League** — Optimizes policy on randomized seeds against an evolving snapshot pool of past checkpoints.

In [ ]:
# 1. Setup & Environment Verification
import os
import sys

# Automatically detect project root regardless of where kernel was launched
cwd = os.getcwd()
PROJECT_ROOT = cwd if os.path.exists(os.path.join(cwd, 'training')) else os.path.abspath(os.path.join(cwd, '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import jax
import jax.numpy as jnp
from flax import nnx
import optax
import numpy as np

print(f"Project Root: {PROJECT_ROOT}")
print(f"JAX Backend: {jax.default_backend()} | Available Devices: {jax.devices()}")

Project Root: c:\Applications and Development\Kaggriculture
JAX Backend: cpu | Available Devices: [CpuDevice(id=0)]


## 2. Abstraction Level 1 Parameterized Actions & Delta Net Worth
Test parameterized 1-turn actions and verify Delta Net Worth ($\Delta\text{NW}$) evaluation.

In [ ]:
from scripts.abs_level_1 import (
    step_farmer_cardinal,
    plant_tile,
    water_tile,
    harvest_tile,
    queue_buy_seed,
    queue_sell,
)
from training.rewards import calculate_net_worth, NetWorthRewardTracker

# Verify 1-step action generation
print("Sample Level 1 Farmer Op:", plant_tile("WHEAT"))
print("Sample Level 1 Market Op:", queue_buy_seed("WHEAT", 2))

# Net Worth Calculation verification
mock_obs = {
    "player": 0,
    "day": 0,
    "farms": [
        {"money": 3000, "tiles": [[None]*10 for _ in range(10)]},
        {"money": 3000, "tiles": [[None]*10 for _ in range(10)]},
    ],
    "private": {"shed": {}, "seeds": {"WHEAT": 2}, "inventories": [{}]},
    "market": {"prices": {"WHEAT": 25}},
}
nw = calculate_net_worth(mock_obs)
print(f"Parsed Farm Net Worth: ${nw:.2f}")

Sample Level 1 Farmer Op: ['PLANT', 'WHEAT']
Sample Level 1 Market Op: ['BUY_SEED', 'WHEAT', 2]
Parsed Farm Net Worth: $3020.00


## 3. Phase 1: Supervised Behavioral Cloning (BC)
Compile Grandmaster offline replay files into state-action pairs and pre-train the ActorCriticNet.

In [ ]:
from training.dataset import load_or_build_dataset, ReplayBatchGenerator
from training.train_bc import train_bc_epoch, evaluate_bc
from model.network import ActorCriticNet

# 1. Load Dataset
dataset = load_or_build_dataset(force_rebuild=False)
num_samples = dataset["own_grid"].shape[0]
print(f"Replay Dataset Size: {num_samples} transitions")

# 2. Initialize Model & Optimizer
rngs = nnx.Rngs(42)
bc_model = ActorCriticNet(rngs=rngs)
optimizer = nnx.Optimizer(bc_model, optax.adamw(learning_rate=1e-3), wrt=nnx.Param)

# 3. Run Behavioral Cloning Epochs (JIT-accelerated with batch_size=256)
train_loader = ReplayBatchGenerator(dataset, batch_size=256, shuffle=True)
for epoch in range(1, 11):
    print("Epoch 1 Started:-")
    loss, actor_loss, critic_loss = train_bc_epoch(bc_model, optimizer, train_loader)
    print(f"Epoch {epoch:02d} | Loss: {loss:.4f} (Actor: {actor_loss:.4f}, Critic: {critic_loss:.4f})")

Loading cached dataset from c:\Applications and Development\Kaggriculture\dataset_cache.npz...
Replay Dataset Size: 161280 transitions
Epoch 1 Started:-
Epoch 01 | Loss: 21.7779 (Actor: 3.1100, Critic: 37.3358)
Epoch 1 Started:-
Epoch 02 | Loss: 11.8153 (Actor: 2.3656, Critic: 18.8994)
Epoch 1 Started:-


## 4. Phase 2: Ghost-Play Reinforcement Learning
Train policy against historical Grandmaster replay opponents on matching episode seeds using PPO and $\Delta\text{NW}$ step rewards.

In [ ]:
from training.train_ghost import run_ghost_play_training

# Train Ghost-Play PPO for 5 episodes
ghost_model = run_ghost_play_training(
    episodes=15,
    steps_per_episode=720,
    learning_rate=3e-4,
    seed=42,
)

## 5. Phase 3: True Self-Play PPO League
Train agent against past snapshot versions of itself across randomized environment seeds.

In [ ]:
from training.train_selfplay import run_selfplay_training

# Run Self-Play League PPO iterations
final_model = run_selfplay_training(
    iterations=20,
    steps_per_episode=720,
    learning_rate=3e-4,
    seed=42,
)

## 6. Evaluation & Benchmark vs Starter Baseline
Evaluate trained model against the competition's built-in `starter` baseline agent.

In [ ]:
from kaggle_environments import make
from model.agent import agent, init_agent

init_agent(seed=42)
env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=False)
env.render(mode="ipython", width=1000, height=700)
env.run([agent, "starter"])
final_step = env.steps[-1]
print(f"Match Result -> Our Agent: ${final_step[0]['reward']} | Starter: ${final_step[1]['reward']}")

In [ ]:
import webbrowser

# Save with UTF-8 encoding and auto-launch in your browser
with open("game.html", "w", encoding="utf-8") as f:
    f.write(env.render(mode="html", width=1000, height=700))

webbrowser.open("game.html")
